# ESPRIT Methods
## Tutorial 9: Rotational Invariance, TLS-ESPRIT, and Unitary ESPRIT

ESPRIT (Roy & Kailath, 1989) estimates DOAs by exploiting the **rotational invariance** of subarray pairs in a ULA — without computing a pseudo-spectrum.

Topics:
1. **Invariance structure** of ULAs
2. **Standard (LS) ESPRIT**
3. **Total Least Squares (TLS) ESPRIT**
4. **Unitary ESPRIT** — real-valued computations
5. **Comparison with MUSIC**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from doa_methods.array_processing import UniformLinearArray, SignalModel
from doa_methods.subspace import MUSIC, ESPRIT, UnitaryESPRIT

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

M = 16
array   = UniformLinearArray(M=M, d=0.5)
sm      = SignalModel(array)
music   = MUSIC(array)
esprit  = ESPRIT(array)
u_esprit = UnitaryESPRIT(array)

doas_true = np.deg2rad([-20.0, 15.0])
K = len(doas_true)
snr_db = 10
N = 200

print("Setup complete.")

## 1. The Rotational Invariance Principle

Divide the $M$-element ULA into two overlapping subarrays:
- $\mathbf{x}_1$: elements $1, 2, \ldots, M-1$ (first subarray)
- $\mathbf{x}_2$: elements $2, 3, \ldots, M$ (second subarray, shifted by $d$)

Their array manifolds are related by:

$$\mathbf{A}_2 = \mathbf{A}_1 \boldsymbol{\Phi}$$

where $\boldsymbol{\Phi} = \text{diag}(e^{-j\omega_1}, \ldots, e^{-j\omega_K})$ and $\omega_k = 2\pi d\sin\theta_k$.

The signal subspaces satisfy the **same invariance**: there exists an invertible $\mathbf{T}$ such that:

$$\mathbf{U}_{s,2} = \mathbf{U}_{s,1} \mathbf{T} \boldsymbol{\Phi} \mathbf{T}^{-1}$$

So the eigenvalues of $\mathbf{T}\boldsymbol{\Phi}\mathbf{T}^{-1}$ — obtained from the subarray relationship — directly give the spatial frequencies.

In [ ]:
X, _, _ = sm.generate_signals(doas_true, N, snr_db, seed=42)
R = X @ X.conj().T / N

eigenvals, eigenvecs = np.linalg.eigh(R)
eigenvals = eigenvals[::-1]; eigenvecs = eigenvecs[:, ::-1]
U_s = eigenvecs[:, :K]    # signal subspace M×K

# Selection matrices for two subarrays
J1 = np.eye(M)[:-1, :]    # first M-1 rows
J2 = np.eye(M)[1:,  :]    # last  M-1 rows

U_s1 = J1 @ U_s    # (M-1) × K
U_s2 = J2 @ U_s    # (M-1) × K

# LS-ESPRIT: solve U_s1 Psi = U_s2
Psi_ls, _, _, _ = np.linalg.lstsq(U_s1, U_s2, rcond=None)
eigenvalues_psi = np.linalg.eigvals(Psi_ls)
omega_ls = -np.angle(eigenvalues_psi)
sin_theta_ls = omega_ls / (2*np.pi * 0.5)
sin_theta_ls = np.clip(np.real(sin_theta_ls), -1, 1)
doas_ls = np.sort(np.arcsin(sin_theta_ls))

print("LS-ESPRIT (manual):")
print(f"  True   : {np.round(np.rad2deg(doas_true), 4)} °")
print(f"  Est    : {np.round(np.rad2deg(doas_ls),   4)} °")

## 2. LS-ESPRIT via the Library

In [ ]:
doas_esprit_ls  = esprit.estimate(X, K=K, ls_method='ls')
doas_esprit_tls = esprit.estimate(X, K=K, ls_method='total')
doas_music_est  = music.estimate(X, K=K)

print(f"True DOAs      : {np.round(np.rad2deg(doas_true),       4)} °")
print(f"LS-ESPRIT      : {np.round(np.rad2deg(doas_esprit_ls),  4)} °")
print(f"TLS-ESPRIT     : {np.round(np.rad2deg(doas_esprit_tls), 4)} °")
print(f"MUSIC          : {np.round(np.rad2deg(doas_music_est),  4)} °")

## 3. TLS vs LS ESPRIT

**LS-ESPRIT** solves $\mathbf{U}_{s,1} \boldsymbol{\Psi} \approx \mathbf{U}_{s,2}$, treating $\mathbf{U}_{s,1}$ as exact.

**TLS-ESPRIT** treats both $\mathbf{U}_{s,1}$ and $\mathbf{U}_{s,2}$ as noisy, minimising the total perturbation.  It uses the SVD of $[\mathbf{U}_{s,1} | \mathbf{U}_{s,2}]$:

$$\text{Minimise} \|[\Delta_1 | \Delta_2]\|_F \quad \text{s.t.} \quad (\mathbf{U}_{s,1}+\Delta_1)\boldsymbol{\Psi} = \mathbf{U}_{s,2}+\Delta_2$$

TLS-ESPRIT is statistically equivalent to Root-MUSIC at high SNR but slightly better at moderate SNR.

In [ ]:
n_trials  = 300
snr_range = np.arange(-5, 26, 3)
N_cmp     = 200
angle_grid_fine = np.linspace(-np.pi/2, np.pi/2, 3601)

def rmse_m(fn, doas, N_, snrs, trials):
    out = []
    for s in snrs:
        errs = []
        for t in range(trials):
            X_, _, _ = sm.generate_signals(doas, N_, s, seed=t)
            try:
                est = np.sort(fn(X_))
                errs.append(np.sqrt(np.mean((est - np.sort(doas))**2)))
            except Exception:
                errs.append(np.pi)
        out.append(np.rad2deg(np.sqrt(np.mean(np.array(errs)**2))))
    return out

rmse_ls  = rmse_m(lambda X_: esprit.estimate(X_, K=K, ls_method='ls'),
                  doas_true, N_cmp, snr_range, n_trials)
rmse_tls = rmse_m(lambda X_: esprit.estimate(X_, K=K, ls_method='total'),
                  doas_true, N_cmp, snr_range, n_trials)
rmse_mu  = rmse_m(lambda X_: music.estimate(X_, K=K, angle_grid=angle_grid_fine),
                  doas_true, N_cmp, snr_range, n_trials)

fig, ax = plt.subplots(figsize=(12, 6))
ax.semilogy(snr_range, rmse_ls,  'g-o',  ms=6, lw=2, label='LS-ESPRIT')
ax.semilogy(snr_range, rmse_tls, 'b-s',  ms=6, lw=2, label='TLS-ESPRIT')
ax.semilogy(snr_range, rmse_mu,  'r-^',  ms=6, lw=2, label='MUSIC')
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('RMSE (°)')
ax.set_title(f'LS-ESPRIT vs TLS-ESPRIT vs MUSIC  (M={M}, N={N_cmp})')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Unitary ESPRIT

**Unitary ESPRIT** transforms the complex signal subspace into a real-valued form using the **unitary transformation** based on the DFT matrix.  Benefits:
- Operates entirely in $\mathbb{R}$ → faster arithmetic
- Often more numerically stable
- Equivalent performance to TLS-ESPRIT asymptotically

In [ ]:
X_u, _, _ = sm.generate_signals(doas_true, N, snr_db, seed=42)
doas_uesprit = u_esprit.estimate(X_u, K=K)

print(f"True DOAs      : {np.round(np.rad2deg(doas_true), 4)} °")
print(f"Unitary ESPRIT : {np.round(np.rad2deg(doas_uesprit), 4)} °")

## 5. Computational Comparison

In [ ]:
import time

N_bench = 500
X_bench, _, _ = sm.generate_signals(doas_true, N_bench, 15, seed=0)
n_rep = 50

methods = {
    'MUSIC (3601-pt grid)': lambda: music.estimate(X_bench, K=K, angle_grid=np.linspace(-np.pi/2,np.pi/2,3601)),
    'LS-ESPRIT'           : lambda: esprit.estimate(X_bench, K=K, ls_method='ls'),
    'TLS-ESPRIT'          : lambda: esprit.estimate(X_bench, K=K, ls_method='total'),
    'Unitary ESPRIT'      : lambda: u_esprit.estimate(X_bench, K=K),
}

print(f"{'Method':<30} {'Time (ms/call)':>18}")
print("-"*50)
for name, fn in methods.items():
    t0 = time.perf_counter()
    for _ in range(n_rep): fn()
    elapsed = (time.perf_counter() - t0) / n_rep * 1000
    print(f"{name:<30} {elapsed:>18.3f}")

## Summary

| Method | Grid needed | Computation | Notes |
|---|---|---|---|
| MUSIC | Yes | $O(M^3) + O(M \cdot N_\theta)$ | Reference subspace method |
| Root-MUSIC | No | $O(M^3) + O(M^2)$ | No grid error |
| LS-ESPRIT | No | $O(M^3) + O(K^3)$ | Fast, slight bias |
| TLS-ESPRIT | No | $O(M^3) + O(K^3)$ | Better accuracy than LS |
| Unitary ESPRIT | No | Same as TLS, real ops | Fastest in practice |

## Exercises
1. Derive the TLS formulation for ESPRIT from first principles (minimising the Frobenius norm of the perturbation in both $\mathbf{U}_{s,1}$ and $\mathbf{U}_{s,2}$).
2. Run a Monte Carlo for 4 sources and compare the RMSE of TLS-ESPRIT vs MUSIC.
3. Implement a variant that uses **three overlapping** subarrays (shift by $d$ and $2d$) to improve robustness — hint: formulate as an overdetermined system.